## Dataset

In [4]:
## Download necessary data files
!mkdir -p data
!cd data
!python -m pip install --force-reinstall gdown
!gdown --folder "https://drive.google.com/drive/folders/1j21cc7-97TedKh_El5E34yI8o5ckI7eK"

# # remove the ones we dont need
# !rm -rf data/pdbbind_v2016
# !rm -rf data/pdbbind_v2020
# !rm data/test_set.zip
# !rm data/crossdocked_v1.1_rmsd1.0.tar.gz

  Using cached gdown-5.2.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached filelock-3.24.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached charset_normalizer-3.4.4-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached gdown-5.2.1-py3-none-any.whl (18 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 

## Create Train/Val/Test Split
The provided dataset from DiffDock has only train and test split. We need to make a validation split from train.

In [1]:
import pickle
import random
from pathlib import Path

import lmdb
import torch

lmdb_path = "data/crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb"
name_split_path = "data/split_by_name.pt"
out_path = "data/crossdocked_pose_split_from_name_val1000.pt"


def pair_key(pf, lf):
    return (str(pf), str(lf))


def pair_key_name(pf, lf):
    return (Path(str(pf)).name, Path(str(lf)).name)


name_split = torch.load(name_split_path, map_location="cpu")
train_pairs = [pair_key(p, l) for p, l in name_split["train"]]
test_pairs = [pair_key(p, l) for p, l in name_split["test"]]

wanted_full = set(train_pairs) | set(test_pairs)
wanted_name = set(pair_key_name(p, l) for p, l in wanted_full)

env = lmdb.open(
    lmdb_path,
    subdir=False,
    readonly=True,
    lock=False,
    readahead=False,
    max_readers=256,
)

mapping = {}
mapping_name = {}

with env.begin() as txn:
    cur = txn.cursor()
    for k, v in cur:
        rec = pickle.loads(v)
        pf = rec.get("protein_filename", "")
        lf = rec.get("ligand_filename", "")
        if not pf or not lf:
            continue

        key_full = pair_key(pf, lf)
        if key_full in wanted_full and key_full not in mapping:
            mapping[key_full] = int(k)

        key_name = pair_key_name(pf, lf)
        if key_name in wanted_name and key_name not in mapping_name:
            mapping_name[key_name] = int(k)

env.close()


def resolve_ids(pairs):
    ids = []
    misses = 0
    for pf, lf in pairs:
        idx = mapping.get((pf, lf))
        if idx is None:
            idx = mapping_name.get(pair_key_name(pf, lf))
        if idx is None:
            misses += 1
            continue
        ids.append(idx)
    return ids, misses


train_ids, train_miss = resolve_ids(train_pairs)
test_ids, test_miss = resolve_ids(test_pairs)

test_set = set(test_ids)
train_ids = [x for x in train_ids if x not in test_set]

seed = 0
val_n = 1000
rng = random.Random(seed)
rng.shuffle(train_ids)

val_ids = train_ids[:val_n]
train_ids = train_ids[val_n:]

if len(val_ids) != val_n:
    raise RuntimeError(f"not enough train ids for val={val_n}: got {len(val_ids)}")

out = {"train": train_ids, "val": val_ids, "test": test_ids}
torch.save(out, out_path)

print(
    "saved:", out_path,
    "\nsizes:", {k: len(out[k]) for k in out},
    "\nmisses:", {"train": train_miss, "test": test_miss},
)

split = torch.load("data/crossdocked_pose_split_from_name_val1000.pt")
assert not (set(split["train"]) & set(split["val"]) &set(split["test"]))

print("no id overlap")

saved: data/crossdocked_pose_split_from_name_val1000.pt 
sizes: {'train': 98990, 'val': 1000, 'test': 100} 
misses: {'train': 10, 'test': 0}
no id overlap


In [1]:
import os
from datamodules import CrossDockedDataModule

lmdb_path = os.path.abspath("data/crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb")
split_path = os.path.abspath("data/crossdocked_pose_split_from_name_val1000.pt")

dm = CrossDockedDataModule(
    lmdb_path=lmdb_path,
    split_pt_path=split_path,
    batch_size=8,
    num_workers=2,
)

batch = next(iter(dm.train_dataloader()))
print("protein_pos:", batch["protein_pos"].shape)
print("ligand_pos :", batch["ligand_pos"].shape)
print("bond_index :", batch["ligand_bond_index"].shape)
print("bond_type  :", batch["ligand_bond_type"].shape)
print("protein_counts:", batch["protein_counts"][:5].tolist())
print("ligand_counts :", batch["ligand_counts"][:5].tolist())
print("keys[0]:", batch["keys"][0])

# print everything within a sample
print("\n--- sample 0 ---")
for k in batch:
    print(f"{k}: {batch[k][0]}")
    break

protein_pos: torch.Size([3533, 3])
ligand_pos : torch.Size([206, 3])
bond_index : torch.Size([2, 450])
bond_type  : torch.Size([450])
protein_counts: [406, 425, 502, 358, 498]
ligand_counts : [25, 26, 33, 12, 22]
keys[0]: b'116191'

--- sample 0 ---
keys: b'116191'


In [3]:
import torch

def median_bond_len(batch):
    pos = batch["ligand_pos"].float()
    e = batch["ligand_bond_index"].long()
    a = pos[e[0]]
    b = pos[e[1]]
    d = (a - b).pow(2).sum(-1).sqrt()
    return float(d.median().item())

def pocket_scale(batch):
    pos = batch["protein_pos"].float()
    return float(torch.pdist(pos, p=2).median().item())

batch = next(iter(dm.train_dataloader()))
print("median bond length:", median_bond_len(batch))
print("median protein pdist:", pocket_scale(batch))


median bond length: 1.4188246726989746
median protein pdist: 14.135181427001953


In [ ]:
# trivial baselines for ligand position prediction from random time step in diffusion model

import torch
from tqdm import tqdm
from model.diffusion import center_by_protein
from model.diffusion import LigandDiffusion
from model.egnn import EGNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

denoiser = EGNN(
    num_layers=6,
    hidden_dim=256,
    edge_feat_dim=4,
    num_r_gaussian=16,
    k=32,
    cutoff_mode="knn",
    update_x=True,
    act_fn="silu",
    norm=False,
).to(device)

ligand_diffusion = LigandDiffusion(denoiser=denoiser, num_types=7).to(device)

@torch.inference_mode()
def pos_baselines(diffusion, loader, device):
    z = 0.0
    xtb = 0.0
    model = 0.0
    n = 0

    for batch in tqdm(loader):
        batch = {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}
        pb = batch["protein_batch"].long()
        lb = batch["ligand_batch"].long()
        bsz = int(pb.max().item()) + 1

        pp = batch["protein_pos"].float()
        lp = batch["ligand_pos"].float()
        pp = pp + diffusion.protein_noise_std * torch.randn_like(pp)
        pp, lp = center_by_protein(pp, lp, pb, lb, bsz)

        t_graph = torch.randint(0, diffusion.t, (bsz,), device=device)
        t_atom = (t_graph + 1)[lb]
        xt, _ = diffusion.q_sample_x(lp, t_atom)

        z += float(lp.pow(2).sum(-1).mean().item()) * bsz
        xtb += float((lp - xt).pow(2).sum(-1).mean().item()) * bsz
        # model += float(diffusion.loss(batch)["loss_position"].item()) * bsz
        n += bsz


    return {"zero": z / n, "xt": xtb / n} #"model": model / n}

print(pos_baselines(ligand_diffusion, dm.val_dataloader(), device))


  0%|          | 0/125 [00:00<?, ?it/s]

100%|██████████| 125/125 [00:02<00:00, 43.15it/s]

{'zero': 26.545380012512208, 'xt': 1.423866429567337}


In [ ]:
# finally, we can delete the leftover files we dont need anymore
!rm data/crossdocked_pocket10_pose_split.pt
!rm data/split_by_name.pt